# EDA (Exploratory Data Analysis)

## Author: 
Yangshunshun Lu
## AI Disclosure Statement: 
This code uses Google Gemini for debugging.
## Description: 
This script performs comprehensive Exploratory Data Analysis (EDA) by merging music audio features with COVID-19 Google Trends data. It generates descriptive statistics, produces time-series and correlation visualizations, and runs OLS regression models to evaluate the baseline relationship between audio features and search trends across multiple geographic regions (All Countries, USA, and Europe).

In [ ]:
# ==========================================
# Phase 1: Setup, Data Loading & Region Mapping
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Global plotting settings and directory setup
sns.set_theme(style="whitegrid", palette="muted")
base_out_dir = os.path.join('..', '..','EDAViz_result', 'EDA_Viz')
out_dir_desc = os.path.join('..', '..', 'EDAViz_result', 'EDA_Viz', 'descriptive_stats')
os.makedirs(base_out_dir, exist_ok=True)
os.makedirs(out_dir_desc, exist_ok=True)

# 1. Load Music Dataset ONCE
print("Loading datasets...")
music_file = os.path.join('..', '..','Data', 'cleaned_data', 'lyricsFeatures_mergeERUS_clean_enonly_token_pos_ldaTfidf.csv')
df_music = pd.read_csv(music_file)

# 2. Apply Region Mapping to Music Data
eu_countries = [
    "Germany", "France", "United Kingdom", "Italy", "Spain", 
    "Netherlands", "Belgium", "Sweden", "Norway", "Denmark", 
    "Finland", "Ireland", "Austria", "Switzerland", "Portugal", 
    "Poland", "Czech Republic"
]

if 'country' in df_music.columns:
    df_music['region'] = np.where(df_music['country'] == 'USA', 'USA',
                          np.where(df_music['country'].isin(eu_countries), 'Europe', 'Other'))
else:
    print("Warning: 'country' column not found in dataset. Region mapping might fail.")

# 3. Load and Format Trends Data
df_trend_eu = pd.read_csv(os.path.join('..', '..', 'Data', 'Google Trends','covid_trends_europe.csv'))
df_trend_us = pd.read_csv(os.path.join('..', '..', 'Data', 'Google Trends','google_trends_us_weekly.csv'))

df_trend_us['country'] = 'USA'
df_trend_us = df_trend_us[['date', 'COVID', 'country']]
df_trend = pd.concat([df_trend_eu, df_trend_us], ignore_index=True)

# 4. Time Transformation
df_music['date'] = pd.to_datetime(df_music['date'], errors='coerce')
df_trend['date'] = pd.to_datetime(df_trend['date'], errors='coerce')

df_music_clean = df_music.dropna(subset=['date']).copy()
df_trend_clean = df_trend.dropna(subset=['date']).copy()

df_music_clean['year_week'] = df_music_clean['date'].dt.strftime('%Y-W%V')
df_trend_clean['year_week'] = df_trend_clean['date'].dt.strftime('%Y-W%V')

df_trend_weekly = df_trend_clean.groupby(['year_week', 'country'])['COVID'].mean().reset_index()

# 5. Create Merged Dataset for EDA
df_merged = pd.merge(df_music_clean, df_trend_weekly, on=['year_week', 'country'], how='inner')
df_merged.rename(columns={'COVID': 'covid_trend'}, inplace=True)

# 6. Split merged data for EDA
df_usa_merged = df_merged[df_merged['region'] == 'USA'].copy()
df_europe_merged = df_merged[df_merged['region'] == 'Europe'].copy()

all_predictors = [
    'text_sentiment', 'danceability', 'energy', 'loudness', 
    'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence'
]

print("Data Loading and Setup Complete.")
print(f"Total Music Records Loaded: {len(df_music)}")
print(f"Merged Dataset (All Countries) shape: {df_merged.shape}")

In [ ]:
# ==========================================
# Phase 2: Descriptive Statistics
# ==========================================
print("\nGenerating Descriptive Statistics...")

# Select only the columns that actually exist in the dataframe to avoid errors
cols_desc = [col for col in all_predictors if col in df_music.columns]

def save_descriptive_stats(df, region_name, out_dir):
    """
    Calculates and exports descriptive statistics for a specified region to a CSV file.
    
    This function computes summary statistics for the features specified in the 
    globally defined `cols_desc` list. If the provided DataFrame contains data, 
    the statistics are saved to the specified output directory. If the DataFrame 
    is empty, it skips the calculation and prints a warning message.

    Args:
        df (pd.DataFrame): The dataset containing the features to be analyzed.
        region_name (str): The identifier for the region (e.g., 'USA', 'Europe'). 
                           This is used to uniquely name the output CSV file.
        out_dir (str): The destination directory path where the CSV file will be saved.

    Returns:
        None
    """
    if len(df) > 0:
        desc_stats = df[cols_desc].describe(include='all')
        out_file = os.path.join(out_dir, f'Table_DescStats_{region_name}.csv')
        desc_stats.to_csv(out_file)
        print(f"   -> {region_name} descriptive stats saved to: {out_file}")
    else:
        print(f"   -> Warning: No data available for {region_name}.")

# Generate stats on the original unmerged music dataset for maximum coverage
save_descriptive_stats(df_music, "All_Countries", out_dir_desc)

if 'region' in df_music.columns:
    save_descriptive_stats(df_music[df_music['region'] == 'USA'], "USA", out_dir_desc)
    save_descriptive_stats(df_music[df_music['region'] == 'Europe'], "Europe", out_dir_desc)

print("Phase 2 Complete.")

In [ ]:
# ==========================================
# Phase 3: Master EDA Function (with Combined Time-Series Grid)
# ==========================================
def run_full_eda(df_region, region_name, base_dir, predictors):
    """
    Executes the full Exploratory Data Analysis (EDA) pipeline for a specific region.
    
    This function generates univariate distributions, time-series overlays, linearity 
    scatter plots, a correlation heatmap, Variance Inflation Factor (VIF) scores, 
    and Ordinary Least Squares (OLS) regression summaries. All visualizations and 
    CSV reports are saved to a dynamically created folder for the specified region.
    
    Args:
        df_region (pd.DataFrame): The dataset specific to the region being analyzed.
        region_name (str): Identifier for the region (e.g., 'USA', 'Europe', 'All_Countries').
        base_dir (str): The root directory where the region's output folder will be created.
        predictors (list): A list of feature column names to analyze (Note: assumes exactly 9 features for 3x3 plotting grids).
        
    Returns:
        None
    """
    print(f"\n" + "="*50)
    print(f"STARTING FULL EDA FOR: {region_name}")
    print("="*50)
    
    out_dir = os.path.join(base_dir, region_name)
    os.makedirs(out_dir, exist_ok=True)

    # ---------------------------------------------------------
    # Step 1: Univariate Analysis (Distributions & Boxplots)
    # ---------------------------------------------------------
    print("1. Generating Distributions & Boxplots...")
    plt.figure(figsize=(10, 5))
    sns.histplot(df_region['covid_trend'], bins=50, kde=True, color='coral')
    plt.title(f'[{region_name}] COVID Search Trend Distribution', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'1_{region_name}_Target_Distribution.png'), dpi=300)
    plt.close()

    plt.figure(figsize=(16, 8))
    standard_features = [f for f in predictors if f != 'loudness']
    sns.boxplot(data=df_region[standard_features], palette="Set3", orient="h")
    plt.title(f'[{region_name}] Features Boxplots', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'2_{region_name}_Features_Boxplots.png'), dpi=300)
    plt.close()

    # ---------------------------------------------------------
    # Step 2: Combined Time-Series Grid
    # ---------------------------------------------------------
    print("2. Generating Combined Time-Series Grid (Master Dashboard)...")
    
    fig, axes = plt.subplots(3, 3, figsize=(24, 18))
    axes = axes.flatten()
    
    for i, feature in enumerate(predictors):
        ax1 = axes[i]
        
        time_agg = df_region.groupby('year_week').agg({
            'covid_trend': 'mean', 
            feature: 'mean'
        }).reset_index()
        time_agg['plot_date'] = pd.to_datetime(time_agg['year_week'].astype(str) + '-0', format='%Y-W%W-%w', errors='coerce')
        time_agg = time_agg.dropna(subset=['plot_date']).sort_values('plot_date')

        color1 = '#e63946'
        ax1.plot(time_agg['plot_date'], time_agg['covid_trend'], color=color1, linewidth=2, label='COVID Trend')
        ax1.set_ylabel('COVID Trend', color=color1, fontsize=12, fontweight='bold')
        ax1.tick_params(axis='y', labelcolor=color1)
        
        ax2 = ax1.twinx()  
        color2 = '#457b9d'
        ax2.plot(time_agg['plot_date'], time_agg[feature], color=color2, linestyle='--', linewidth=2.5, label=feature.capitalize())
        ax2.set_ylabel(feature.capitalize(), color=color2, fontsize=12, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor=color2)
        
        ax1.set_title(f'{feature.capitalize()} vs. COVID Trend', fontsize=16, fontweight='bold')
        ax1.tick_params(axis='x', rotation=30) 

    plt.tight_layout()
    plt.subplots_adjust(top=0.92) 
    fig.suptitle(f'[{region_name}] Temporal Evolution: Music Features vs. COVID Trend', fontsize=26, fontweight='bold')
    
    grid_path = os.path.join(out_dir, f'3_{region_name}_Combined_TimeSeries_Grid.png')
    plt.savefig(grid_path, dpi=300)
    plt.close()
    print(f"   -> Master Dashboard saved: {grid_path}")

    # ---------------------------------------------------------
    # Step 3: Linearity (Scatter Grid) & Correlation Heatmap
    # ---------------------------------------------------------
    print("3. Checking Linearity & Multicollinearity...")
    plot_sample = df_region.sample(n=min(3000, len(df_region)), random_state=42)
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    axes = axes.flatten()
    for i, feature in enumerate(predictors):
        if feature in df_region.columns:
            sns.regplot(data=plot_sample, x=feature, y='covid_trend', ax=axes[i], scatter_kws={'alpha': 0.15, 's': 10}, line_kws={'color': 'red'})
            axes[i].set_title(f'Trend vs {feature.capitalize()}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'4_{region_name}_Linearity_Grid.png'), dpi=300)
    plt.close()

    plt.figure(figsize=(11, 9))
    corr_matrix = df_region[predictors].corr(method='spearman')
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=.5, cbar_kws={"shrink": .8})
    plt.title(f'[{region_name}] Correlation Matrix', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'5_{region_name}_Correlation_Heatmap.png'), dpi=300)
    plt.close()

    # ---------------------------------------------------------
    # Step 4: VIF Calculation
    # ---------------------------------------------------------
    print("4. Calculating VIF...")
    vif_data_clean = df_region[predictors].dropna()
    X_vif = sm.add_constant(vif_data_clean)
    vif_df = pd.DataFrame({"Feature": X_vif.columns, "VIF_Score": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]})
    vif_df = vif_df[vif_df['Feature'] != 'const'].sort_values('VIF_Score', ascending=False)
    vif_csv_path = os.path.join(out_dir, f'6_{region_name}_VIF_Results.csv')
    vif_df.to_csv(vif_csv_path, index=False)

    # ---------------------------------------------------------
    # Step 5: Baseline Exploratory Statistical Testing
    # ---------------------------------------------------------
    print("5. Running Standard OLS (sm.OLS) and Exporting Summaries...")
    
    baseline_features = ['text_sentiment', 'valence', 'danceability', 'energy', 'acousticness', 'instrumentalness', 'liveness']
    y_col = 'covid_trend'
    
    for feature in baseline_features:
        if feature in df_region.columns:
            clean_data = df_region[[y_col, feature]].dropna()
            
            y = clean_data[y_col]
            x = clean_data[feature]
            x_with_const = sm.add_constant(x)
            
            ols_m = sm.OLS(y, x_with_const)
            results = ols_m.fit()
            
            summary_csv_path = os.path.join(out_dir, f'7_{region_name}_OLS_Summary_{feature}.csv')
            
            with open(summary_csv_path, 'w') as f:
                f.write(results.summary().as_csv())

    print(f"   -> All OLS summaries exported successfully.")
    print(f"\n{region_name} EDA Pipeline Complete.")

In [ ]:
# ==========================================
# Phase 4: Execute the Pipeline for All Countries, USA, and Europe
# ==========================================

print("Initiating EDA processing pipeline...")

# 1. Run for All Countries (Full Dataset)
run_full_eda(df_region=df_merged, region_name="All_Countries", base_dir=base_out_dir, predictors=all_predictors)

# 2. Run for USA
run_full_eda(df_region=df_usa_merged, region_name="USA", base_dir=base_out_dir, predictors=all_predictors)

# 3. Run for Europe
run_full_eda(df_region=df_europe_merged, region_name="Europe", base_dir=base_out_dir, predictors=all_predictors)

print("\nProcessing finished. Please check the EDA folder.")